# Curvature-Coupled Dark Energy: Background and Diagnostic Figures

This notebook reproduces the background and closely related diagnostic plots used in the
**Curvature-Coupled Dark Energy (CCDE)** paper.

The notebook mainly:
1. loads the precomputed `hi_class` background outputs,
2. selects the representative $(\sigma,\alpha)$ models used in the paper,
3. constructs the plotted background/EFT quantities, and
4. produces the corresponding figures.



In [ ]:

# Standard library
import os
from os.path import exists
from collections import defaultdict

# Numerical / plotting libraries
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator


def nested_dict(n, value_type):
    """Create an n-level nested defaultdict."""
    if n == 1:
        return defaultdict(value_type)
    return defaultdict(lambda: nested_dict(n - 1, value_type))


# Container for the loaded hi_class outputs.
data = nested_dict(5, list)


# ------------------------------------------------------------------
# hi_class background-output columns (Python uses zero-based indexing)
# ------------------------------------------------------------------
COL_Z = 0
COL_H = 3

COL_RHO_G = 8
COL_RHO_B = 9
COL_RHO_CDM = 10
COL_RHO_NCDM = 11
COL_P_NCDM = 12
COL_RHO_UR = 13
COL_RHO_CRIT = 14
COL_RHO_TOT = 15
COL_P_TOT = 16
COL_P_TOT_PRIME = 17

COL_GROWTH_D = 18
COL_GROWTH_F = 19

COL_RHO_PHI = 20
COL_P_PHI = 21
COL_MSTAR2 = 22
COL_DMSTAR2 = 23
COL_ALPHA_K = 24
COL_ALPHA_B = 25
COL_ALPHA_T = 26
COL_ALPHA_M = 27
COL_ALPHA_H = 28
COL_CS2 = 29

COL_PHI = 33
COL_PHI_PRIME = 34
COL_PHI_PP = 35

COL_GEFF = 46
COL_SLIP_EFF = 47


## 1. Load the CCDE background outputs

This cell defines the parameter grid and loads the precomputed `hi_class` background files.

The paper uses the shifted-quartic curvature coupling and inverse-power-law potential,
parameterized by $(\alpha,\sigma)$. The files are stored by parameter pair as

`run_sigma<sigma>_alpha<alpha>/CCDE_sigma<sigma>_alpha<alpha>_background.dat`.

A separate $\Lambda$CDM background is loaded as the reference model. Strings are kept
exactly as they appear in the directory names so that file lookup is deterministic.


In [ ]:

# Redshift sampling retained from the original analysis.
redshifts = [
    100, 50, 30, 20, 10, 6, 5, 4, 3, 2.5, 2, 1.75, 1.5, 1.25, 1.1, 1.0,
    0.9, 0.8, 0.75, 0.7, 0.6, 0.5, 0.4, 0.3, 0.25, 0.2, 0.1, 0.0,
]

data_address = "./../DataGenerator/transfer_functions/"
load_data = True

# Strings are used so that directory/file names are matched exactly.
alphas = ["0.1", "0.05", "0.001", "-0.05", "-0.1"]
sigmas = ["0.001", "0.3", "1.", "1.5"]

alpha = np.array(sorted(set(alphas), key=float), dtype=object)
sigma = np.array(sorted(set(sigmas), key=float), dtype=object)

alpha_to_j = {a: j for j, a in enumerate(alpha)}
sigma_to_i = {s: i for i, s in enumerate(sigma)}

# Keep the original 2D run-label array used by the analysis.
out_arr = np.empty((len(sigma), len(alpha)), dtype=object)

data.setdefault("bg_data", {})
for s in sigma:
    data["bg_data"].setdefault(f"sigma={s}", {})
    for a in alpha:
        data["bg_data"][f"sigma={s}"].setdefault(f"alpha={a}", None)

# Load the reference LambdaCDM background.
lcdm_bg = os.path.join(data_address, "LCDM", "LCDM_background.dat")
lcdm_sigma = "0.0"
lcdm_alpha = "0.0"

data["bg_data"].setdefault(f"sigma={lcdm_sigma}", {})
data["bg_data"][f"sigma={lcdm_sigma}"].setdefault(f"alpha={lcdm_alpha}", None)

if exists(lcdm_bg):
    print("\033[94mLoading LCDM as alpha=0, sigma=0\033[0m")
    data["bg_data"][f"sigma={lcdm_sigma}"][f"alpha={lcdm_alpha}"] = np.loadtxt(lcdm_bg)
else:
    print(lcdm_bg, "\033[91mLCDM file not found!\033[0m")

# Load the CCDE parameter grid.
total_loaded = 0
missing = []

if load_data:
    for a_str in alpha:
        for s_str in sigma:
            run_tag = f"sigma{s_str}_alpha{a_str}"
            base_path = os.path.join(data_address, "run_" + run_tag)
            bg_file = os.path.join(base_path, "CCDE_" + run_tag + "_background.dat")

            i = sigma_to_i[s_str]
            j = alpha_to_j[a_str]
            out_arr[i, j] = run_tag

            if exists(bg_file):
                print(f"\033[94m{run_tag} exists — loading\033[0m")
                data["bg_data"][f"sigma={s_str}"][f"alpha={a_str}"] = np.loadtxt(bg_file)
                total_loaded += 1
            else:
                missing.append(bg_file)

print("Number of CCDE simulations loaded:", total_loaded)
print("Alphas loaded:", list(alpha))
print("Sigmas loaded:", list(sigma))

if missing:
    print(f"\033[93mMissing {len(missing)} files (showing up to 10):\033[0m")
    for filename in missing[:10]:
        print("  -", filename)


## 2. Representative models used in the paper

The same nine representative $(\sigma,\alpha)$ combinations are reused in the plots below.
Their ordering is kept fixed so that each model has a consistent colour across figures.


## 3. Effective Planck mass and EFT $\alpha$-functions — paper Fig. 5

This figure shows the background quantities used to characterize the linear Horndeski/EFT sector:

- the effective Planck mass, $M_*^2/M_{\rm P}^2 = 1+f(\varphi)$,
- the kineticity, $\alpha_{\rm K}$,
- the Planck-mass run rate, $\alpha_{\rm M}$.

The notebook reads $\alpha_{\rm M}$ directly from the `hi_class` background output. For this model,
the analytic relation $\alpha_{\rm M}=-\alpha_{\rm B}$ holds.


In [ ]:

# ---- Plot style parameters ----
text_size = 30
fig_size_x = 28
fig_size_y = 10  # a bit shorter for 1×3
lw_f = 3.

plt.rc('text', usetex=True)
font = {'family': 'normal', 'weight': 'bold', 'size': text_size}
plt.rc('font', **font)

# ---- Figure: 1 row, 3 columns ----
fig, axes = plt.subplots(
    1, 3,
    sharex=True, sharey=False,
    figsize=(fig_size_x, fig_size_y),
    facecolor='w'
)
ax_Mstar2, ax_alphaK, ax_alphaM = axes
plt.subplots_adjust(wspace=0.25)

# Fixed paper-wide colour convention: keep the same model-to-colour mapping across all notebooks.
colors = [
    '#000000',  # black (if you want to keep it)
    '#0072B2',  # blue
    '#E69F00',  # orange
    '#009E73',  # green
    '#D55E00',  # vermillion
    '#56B4E9',  # sky blue
    '#CC79A7',  # purple
    '#F0E442',  # yellow
    '#882255',  # wine red
    '#AA4499',  # purple
]


# alphas = ["0.1", "0.05", "0.001", "-0.05", "-0.1"]
# sigmas = ["0.001", "0.3", "1.", "1.5"]  # use the sigma grid (from the folders)

# ---- Model choices ----
alpha_consts = ["0.001", "0.1", "0.001", "-0.1", "0.1", "-0.1", "0.1", "0.001", "-0.05"]
sigma_consts = ["0.001", "0.3", "0.3", "0.3", "1.", "1.", "1.5", "1.5", "1.5"]


for num, (sigma_l, alpha_l) in enumerate(zip(sigma_consts, alpha_consts)):
    sigma_key = f"sigma={sigma_l}"
    alpha_key = f"alpha={alpha_l}"

    arr = data['bg_data'].get(sigma_key, {}).get(alpha_key, None)
    if arr is None:
        print("Missing/None:", sigma_key, alpha_key)
        continue

    z1 = arr[:, COL_Z] + 1.0

    M_s     = arr[:, COL_MSTAR2]
    alpha_k = arr[:, COL_ALPHA_K]
    alpha_b = arr[:, COL_ALPHA_B]
    # alpha_M = -alpha_b
    alpha_M = arr[:, COL_ALPHA_M]
    label = rf'$(\sigma={float(sigma_l):g},\,\alpha={float(alpha_l):g})$'
    color = colors[num % len(colors)]

    ax_Mstar2.semilogx(z1, M_s, '-', c=color, lw=lw_f)
    ax_alphaK.semilogx(z1, alpha_k, '-', c=color, lw=lw_f, label=label)
    ax_alphaM.semilogx(z1, alpha_M, '-', c=color, lw=lw_f)

# ---- Axes formatting ----
for ax in axes:
    ax.set_xlim(0.9, 500)
    ax.grid(True, axis='both')

for ax in axes:
    ax.tick_params(which='both', direction='in', top=True, right=True)
    ax.grid(True, which='major', alpha=0.35)
    ax.grid(True, which='minor', alpha=0.15)

ax_Mstar2.set_xlabel(r'$1+z$')
ax_alphaK.set_xlabel(r'$1+z$')
ax_alphaM.set_xlabel(r'$1+z$')
# ax_alphaM.set_xlim(1, 10000)
ax_Mstar2.set_ylabel(r'$M_*^2(z)/M_{\rm P}^2$')
ax_alphaK.set_ylabel(r'$\alpha_{\mathrm{K}}(z)$')
ax_alphaM.set_ylabel(r'$\alpha_{\mathrm{M}}(z) = -\alpha_{\mathrm{B}}(z)$')

# Legend only on the first axis
ax_alphaK.legend(loc='best', frameon=True, fontsize=22.5)

# Save / show
plt.savefig('./Figs/Ms_alphaK_alphaM.pdf', format='pdf', dpi=300,
            bbox_inches='tight', pad_inches=0.1)
plt.show()

# Note: alpha_M is read directly from the stored background output; analytically alpha_M = -alpha_B.


## 4. Scalar-field evolution and background energy budget — paper Fig. 1

This figure shows:

- **Left:** the Planck-normalized scalar amplitude, $\varphi/M_{\rm P}$.
- **Middle:** the field variation per e-fold,
  $|d\varphi/d\ln a|/M_{\rm P}=|\varphi'|/(\mathcal{H}M_{\rm P})$,
  where $\mathcal{H}=aH$. Solid and dashed segments indicate
  $\varphi'>0$ and $\varphi'<0$, respectively.
- **Right:** the fractional densities $\Omega_\varphi$, $\Omega_{\rm m}$, and $\Omega_{\rm r}$.

For the single massive-neutrino species, the background contribution is split smoothly into
radiation-like and matter-like pieces using
$\rho_{\nu,\rm rel}=3P_\nu$ and
$\rho_{\nu,\rm nr}=\rho_\nu-3P_\nu$.
This reproduces the relativistic and non-relativistic limits while keeping the total neutrino
energy density unchanged.


In [ ]:
# ---- Plot style parameters ----
text_size = 30
fig_size_x = 28
fig_size_y = 10

plt.rc('text', usetex=True)
font = {'family': 'normal', 'weight': 'bold', 'size': text_size}
plt.rc('font', **font)

# ---- Figure: 1 row, 3 columns ----
fig, axes = plt.subplots(
    1, 3,
    sharex=False,
    sharey=False,
    figsize=(fig_size_x, fig_size_y),
    facecolor='w'
)

ax_phi, ax_dphi, ax_Om = axes
plt.subplots_adjust(wspace=0.25)


for num, (sigma_l, alpha_l) in enumerate(zip(sigma_consts, alpha_consts)):

    sigma_key = f"sigma={sigma_l}"
    alpha_key = f"alpha={alpha_l}"

    if (
        sigma_key not in data['bg_data']
        or alpha_key not in data['bg_data'][sigma_key]
    ):
        continue

    arr = data['bg_data'][sigma_key][alpha_key]

    if arr is None:
        continue

    # --------------------------------------------------------------
    # Background quantities
    # --------------------------------------------------------------

    # x-axis: 1 + z
    z1 = arr[:, COL_Z] + 1.0
    a = 1.0 / z1

    # Physical Hubble parameter H.
    # The conformal Hubble parameter is H_conf = a H.
    H = arr[:, COL_H]
    H_conf = a * H

    # Scalar field and its conformal-time derivative.
    # hi_class_CCDE stores phi' = dphi/deta directly.
    phi_smg = arr[:, COL_PHI]
    phi_prime = arr[:, COL_PHI_PRIME]

    # Evolution of the scalar field per e-fold:
    #
    #     dphi/dln(a) = phi' / H_conf = phi' / (a H)
    #
    # This directly measures the field evolution relative to the
    # cosmological expansion.
    dphi_dlna = phi_prime / H_conf

    # --------------------------------------------------------------
    # Background densities
    # --------------------------------------------------------------

    rho_g     = arr[:, COL_RHO_G]
    rho_b     = arr[:, COL_RHO_B]
    rho_cdm   = arr[:, COL_RHO_CDM]

    rho_ncdm  = arr[:, COL_RHO_NCDM]
    p_ncdm    = arr[:, COL_P_NCDM]

    rho_ur    = arr[:, COL_RHO_UR]
    rho_smg   = arr[:, COL_RHO_PHI]
    rho_crit  = arr[:, COL_RHO_CRIT]

    # --------------------------------------------------------------
    # Split the massive-neutrino component into effective
    # relativistic and non-relativistic pieces.
    #
    # For a relativistic species:
    #     P = rho/3  -> rho_ncdm_rel = rho_ncdm
    #
    # For a non-relativistic species:
    #     P -> 0     -> rho_ncdm_nr = rho_ncdm
    #
    # Thus the decomposition smoothly interpolates between the
    # radiation-like and matter-like regimes.
    # --------------------------------------------------------------

    rho_ncdm_rel = 3.0 * p_ncdm
    rho_ncdm_nr  = rho_ncdm - rho_ncdm_rel

    label = rf'$(\sigma={float(sigma_l):g},\,\alpha={float(alpha_l):g})$'


    # --------------------------------------------------------------
    # Left panel: scalar-field amplitude phi/M_P
    # --------------------------------------------------------------

    ax_phi.semilogx(
        z1,
        phi_smg,
        '-',
        c=colors[num],
        lw=lw_f,
        label=label
    )


    # --------------------------------------------------------------
    # Middle panel: |dphi/dln(a)|
    #
    # Solid  : phi' > 0
    # Dashed : phi' < 0
    #
    # Keep the full redshift array and use NaNs to break the curve
    # at sign changes. This avoids connecting disconnected branches.
    # --------------------------------------------------------------

    abs_dphi_dlna = np.abs(dphi_dlna)

    dphi_positive = np.where(
        dphi_dlna > 0,
        abs_dphi_dlna,
        np.nan
    )

    dphi_negative = np.where(
        dphi_dlna < 0,
        abs_dphi_dlna,
        np.nan
    )

    ax_dphi.loglog(
        z1,
        dphi_positive,
        '-',
        c=colors[num],
        lw=lw_f
    )

    ax_dphi.loglog(
        z1,
        dphi_negative,
        '--',
        c=colors[num],
        lw=lw_f
    )


    # --------------------------------------------------------------
    # Right panel: fractional background densities
    # --------------------------------------------------------------

    Om_smg = rho_smg / rho_crit

    # Matter:
    # baryons + CDM + non-relativistic part of the massive neutrino
    Om_m = (
        rho_b
        + rho_cdm
        + rho_ncdm_nr
    ) / rho_crit

    # Radiation:
    # photons + ultra-relativistic species + relativistic part of
    # the massive neutrino
    Om_r = (
        rho_g
        + rho_ur
        + rho_ncdm_rel
    ) / rho_crit

    # Add the density-component labels only once.
    if num == 0:

        ax_Om.semilogx(
            z1,
            Om_smg,
            ':',
            c='k',
            lw=lw_f,
            label=r'$\Omega_\varphi$'
        )

        ax_Om.semilogx(
            z1,
            Om_m,
            '-',
            c='k',
            lw=lw_f,
            label=r'$\Omega_{\rm m}$'
        )

        ax_Om.semilogx(
            z1,
            Om_r,
            '-.',
            c='k',
            lw=lw_f,
            label=r'$\Omega_{\rm r}$'
        )

    # Model-dependent coloured curves
    ax_Om.semilogx(
        z1,
        Om_smg,
        ':',
        c=colors[num],
        lw=lw_f
    )

    ax_Om.semilogx(
        z1,
        Om_m,
        '-',
        c=colors[num],
        lw=lw_f
    )

    ax_Om.semilogx(
        z1,
        Om_r,
        '-.',
        c=colors[num],
        lw=lw_f
    )


# ------------------------------------------------------------------
# Formatting
# ------------------------------------------------------------------

for ax in axes:

    ax.set_xlim(1, 1e7)

    ax.tick_params(
        which='both',
        direction='in',
        top=True,
        right=True
    )

    ax.grid(True, which='major', alpha=0.35)
    ax.grid(True, which='minor', alpha=0.15)


# ---- Left panel
ax_phi.set_xlabel(r'$1+z$')
ax_phi.set_ylabel(r'$\varphi(z)/M_{\rm P}$')
ax_phi.set_ylim(-0.1, 1.75)


# ---- Middle panel
ax_dphi.set_xlabel(r'$1+z$')
ax_dphi.set_ylabel(
    r'$\left|d\varphi/d\ln a\right|/M_{\rm P}$'
)

ax_dphi.set_ylim(1.e-8, 20)


# ---- Right panel
ax_Om.set_xlabel(r'$1+z$')
ax_Om.set_ylabel(r'$\Omega_i(z)$')
ax_Om.set_ylim(-0.25, 1.5)


# ------------------------------------------------------------------
# Legends
# ------------------------------------------------------------------

# Since aH > 0, dphi/dln(a) and phi' have the same sign.
sign_legend = [
    Line2D(
        [0], [0],
        color='k',
        lw=lw_f,
        linestyle='-',
        label=r'$\varphi^\prime>0$'
    ),
    Line2D(
        [0], [0],
        color='k',
        lw=lw_f,
        linestyle='--',
        label=r'$\varphi^\prime<0$'
    ),
]

ax_dphi.legend(
    handles=sign_legend,
    loc='upper right',
    frameon=True,
    fontsize=27
)

# Model legend
ax_phi.legend(
    frameon=True,
    fontsize=22
)

# Density-component legend
ax_Om.legend(
    frameon=True,
    fontsize=26
)


# ------------------------------------------------------------------
# Save figure
# ------------------------------------------------------------------

plt.savefig(
    './Figs/background_phi_dphiDlna_Omega.pdf',
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.1
)

plt.show()

## 5. Effective scalar density, pressure, and equation of state — paper Fig. 2

The effective scalar-fluid quantities are read directly from the `hi_class` background output,
with
\[
w_\varphi(z)=\frac{P_\varphi(z)}{\rho_\varphi(z)}.
\]

Because $\rho_\varphi$ and $P_\varphi$ can change sign, the first two panels show their
absolute values on logarithmic axes, with solid curves for positive values and dashed curves
for negative values. When $\rho_\varphi$ crosses zero, $w_\varphi$ develops the pole discussed
in the paper.


In [ ]:

# ---- Plot style parameters ----
text_size = 30
fig_size_x = 28
fig_size_y = 10

plt.rc('text', usetex=True)
plt.rc('font', family='normal', weight='bold', size=text_size)

# ---- Helper: signed values on a log-y axis ----
def plot_signed_logy(ax, x, y, *, color, lw=lw_f, label=None):
    """Plot signed data on a log-y axis by drawing |y| and encoding sign in linestyle.

    Solid:  y>0 plotted as y
    Dashed: y<0 plotted as |y|
    """
    x = np.asarray(x)
    y = np.asarray(y)

    good = np.isfinite(x) & np.isfinite(y) & (x > 0)
    xg, yg = x[good], y[good]

    pos = yg > 0
    neg = yg < 0

    if np.any(pos):
        ax.semilogx(xg[pos], yg[pos], '-', color=color, lw=lw_f, label=label)
    else:
        # keep legend entry even if no positive values for this model
        if label is not None:
            ax.plot([], [], '-', color=color, lw=lw_f, label=label)

    if np.any(neg):
        ax.semilogx(xg[neg], -yg[neg], '--', color=color, lw=lw_f)

# ---- Figure: 1 row, 3 columns ----
fig, axes = plt.subplots(
    1, 3,
    sharex=False, sharey=False,
    figsize=(fig_size_x, fig_size_y),
    facecolor='w'
)
ax_rho, ax_p, ax_w = axes
plt.subplots_adjust(wspace=0.25)


for num, (sigma_l, alpha_l) in enumerate(zip(sigma_consts, alpha_consts)):
    sigma_key = f"sigma={sigma_l}"
    alpha_key = f"alpha={alpha_l}"

    arr = data['bg_data'].get(sigma_key, {}).get(alpha_key, None)
    if arr is None:
        print("Missing/None:", sigma_key, alpha_key)
        continue

    # x-axis: 1+z
    z1 = arr[:, COL_Z] + 1.0

    # Background columns (adjust if needed)
    rho_smg = arr[:, COL_RHO_PHI]   # rho_phi
    p_smg   = arr[:, COL_P_PHI]   # p_phi
    w_smg     = arr[:, COL_P_PHI] / arr[:, COL_RHO_PHI]  # w = p_smg/rho_smg
    label = rf'$(\sigma={float(sigma_l):g},\,\alpha={float(alpha_l):g})$'
    color = colors[num % len(colors)]

    # Left: |rho_phi| with sign encoded by linestyle
    plot_signed_logy(ax_rho, z1, rho_smg, color=color, lw=lw_f)

    # Middle: |p_phi| with sign encoded by linestyle + model label for legend
    plot_signed_logy(ax_p, z1, p_smg, color=color, lw=lw_f, label=label)

    # Right panel: w_phi(z)    
    w_plot = w_smg.copy()
    mask = (w_plot <= -2.5) | (w_plot >= 2.5)
    # mask = np.zeros(w_plot.shape, dtype=bool)
    w_plot[mask] = np.nan
    ax_w.semilogx(z1, w_plot, '-', c=colors[num], lw=lw_f)
    
    # Right: G_eff (typically positive)
    # ax_Geff.semilogx(z1, G_eff, '-', color=color, lw=3.5)

# Reference values for the equation of state.
ax_w.axhline(-1.0, color='k', linestyle='-.', lw=1.0, alpha=0.6)
ax_w.axhline(0.0, color='k', linestyle='-.', lw=1.0, alpha=0.3)
ax_w.axhline(1.0 / 3.0, color='k', linestyle='-.', lw=1.0, alpha=0.3)

# ---- Axes formatting ----
for ax in axes:
    # ax.set_xlim(1, 1e3)
    ax.tick_params(which='both', direction='in', top=True, right=True)
    ax.grid(True, which='major', alpha=0.35)
    ax.grid(True, which='minor', alpha=0.15)

ax_rho.set_xlabel(r'$1+z$')
ax_p.set_xlabel(r'$1+z$')
ax_w.set_xlabel(r'$1+z$')
ax_rho.set_yscale('log')
ax_p.set_yscale('log')

# Tune these to your preferred range (kept from your original cell)
ax_rho.set_ylim(1.e-9, 150)
ax_p.set_ylim(1.e-9, 150)
ax_p.set_xlim(1, 1e3)
ax_rho.set_xlim(1, 1e3)

ax_rho.set_ylabel(r'$|\rho_{\varphi}(z)|\,[{\rm Mpc}^{-2}]$')
ax_p.set_ylabel(r'$|P_{\varphi}(z)|\,[{\rm Mpc}^{-2}]$')
# ax_w.set_ylabel(r'$G_{\rm eff}(z)$')
ax_w.set_xlabel(r'$1+z$')
ax_w.set_ylim(-1.8, 1.0)
ax_w.set_xlim(1, 1e5)
ax_w.set_ylabel(r'$w_{\varphi}(z)$')

# Model legend (use the middle panel, since it carries the labels)
ax_p.legend(loc='upper right', bbox_to_anchor=(0.67, 1), frameon=True, fontsize=22.5, borderaxespad=0.1)

# Sign convention legend (applies to both rho and p)
sign_legend = [
    Line2D([0], [0], color='k', lw=lw_f, linestyle='-',  label=r'positive'),
    Line2D([0], [0], color='k', lw=lw_f, linestyle='--', label=r'negative (plotted as $|\cdot|$)'),
]
ax_rho.legend(handles=sign_legend, loc='upper left', frameon=True, fontsize=27)

plt.savefig('./Figs/rho_P_w.pdf', format='pdf', dpi=300,
            bbox_inches='tight', pad_inches=0.1)
plt.show()


## 6. Background $G_{\rm eff}$ and growth-rate diagnostic

This diagnostic reads the quasistatic $G_{\rm eff}/G_{\rm N}$ and the background growth-rate quantity directly from the background output. It is **not** the same calculation as paper Fig. 6, which reconstructs $\mu_\Psi(k,z)$ and the scale-dependent growth rate from the full perturbation transfer functions.


In [ ]:
# Diagnostic: background G_eff/G_N and stored growth-rate quantity.
#
# This is a useful consistency diagnostic, but it is not identical to paper Fig. 6.
# Fig. 6 reconstructs mu_Psi(k,z) and the scale-dependent growth rate from the full
# perturbation transfer functions at k = 10 Mpc^{-1}. Here G_eff is read directly
# from the background output (the analytic quasistatic quantity).


# ---- Plot style parameters ----
text_size = 30
fig_size_x = 20
fig_size_y = 7.5

plt.rc('text', usetex=True)
font = {'family': 'normal', 'weight': 'bold', 'size': text_size}
plt.rc('font', **font)

# ---- Figure: 1 row, 2 columns ----
fig, axes = plt.subplots(
    1, 2,
    figsize=(fig_size_x, fig_size_y),
    facecolor='w'
)

ax_Geff, ax_f = axes

plt.subplots_adjust(wspace=0.15, hspace=0.05)

COL_Z = 0
COL_Geff = 46
COL_f_default = 19
COL_f_lcdm = 20

baseline_sigma = "0.0"
baseline_alpha = "0.0"

for num,(sigma_l,alpha_l) in enumerate(zip(sigma_consts,alpha_consts)):

    sigma_key=f"sigma={sigma_l}"
    alpha_key=f"alpha={alpha_l}"

    if sigma_key not in data['bg_data'] or alpha_key not in data['bg_data'][sigma_key]:
        continue

    arr=data['bg_data'][sigma_key][alpha_key]

    z1=arr[:,COL_Z]+1

    if (sigma_l==baseline_sigma) and (alpha_l==baseline_alpha):
        Geff=z1/z1
        f_growth=arr[:,COL_f_lcdm]
        label=r'$\Lambda{\rm CDM}$'
    else:
        Geff=arr[:,COL_Geff]
        f_growth=arr[:,COL_f_default]
        label=rf'$(\sigma={float(sigma_l):g},\,\alpha={float(alpha_l):g})$'

    ax_Geff.semilogx(z1,Geff,'-',c=colors[num],lw=lw_f)
    ax_f.semilogx(z1,f_growth,'-',c=colors[num],lw=lw_f,label=label)

# ---- Formatting
for ax in axes:
    ax.set_xlim(1,1e3)
    ax.tick_params(which='both',direction='in',top=True,right=True)
    ax.grid(True,which='major',alpha=0.35)
    ax.grid(True,which='minor',alpha=0.15)

ax_Geff.set_xlabel(r'$1+z$')
ax_Geff.set_ylabel(r'$G_{\rm eff}(z)/G_{\rm N}$')

ax_f.yaxis.set_major_locator(MultipleLocator(0.2))
ax_f.yaxis.set_minor_locator(MultipleLocator(0.1))
ax_Geff.yaxis.set_major_locator(MultipleLocator(0.1))
ax_Geff.yaxis.set_minor_locator(MultipleLocator(0.05))

ax_f.set_xlabel(r'$1+z$')
ax_f.set_ylabel(r'$f(z)$')
ax_f.set_ylim(0.47,2.0)
ax_Geff.set_ylim(0.80,1.9)
# ax_f.legend(loc='upper right',frameon=False,fontsize=22, ncol=2)
ax_f.legend(loc='upper left', frameon=False, bbox_to_anchor=(0.0,1.01), fontsize=21, ncol=2, columnspacing=0.25)
plt.savefig('./Figs/Geff_f.pdf',
            dpi=300,bbox_inches='tight',pad_inches=0.1)
plt.show()


## 7. Scalar-derivative diagnostic

This additional check displays $\varphi$, $\varphi'$, and $\varphi''$ from the background output. The stored prime quantities are conformal-time derivatives and are used directly, with no additional factor of $a$. This diagnostic is not a numbered figure in the paper.


In [ ]:
# Additional scalar-field derivative diagnostic (not a numbered paper figure).
#
# This cell inspects phi, phi', and phi'' over the full stored redshift range.
# Header already labels the stored derivatives as phi' and phi'', so no extra factor of a is applied.
# It is retained as a numerical/background-evolution diagnostic rather than as
# part of the main paper-figure sequence.


# ---- Plot style parameters ----
text_size = 30
fig_size_x = 28
fig_size_y = 10  # a bit shorter for 1×3


plt.rc('text', usetex=True)
font = {'family': 'normal', 'weight': 'bold', 'size': text_size}
plt.rc('font', **font)

# ---- Figure: 1 row, 3 columns ----
fig, axes = plt.subplots(
    1, 3,
    sharex=False, sharey=False,
    figsize=(fig_size_x, fig_size_y),
    facecolor='w'
)

# Diagnostic panels: phi/M_P, |phi'|/(H0 M_P), and |phi''|/(H0^2 M_P)
ax_phiH0, ax_phiPrimeAbs, ax_phi_pp = axes

plt.subplots_adjust(wspace=0.25)


for num, (sigma_l, alpha_l) in enumerate(zip(sigma_consts, alpha_consts)):
    sigma_key = f"sigma={sigma_l}"
    alpha_key = f"alpha={alpha_l}"

    if sigma_key not in data['bg_data'] or alpha_key not in data['bg_data'][sigma_key]:
        continue

    arr = data['bg_data'][sigma_key][alpha_key]

    # x-axis: 1+z
    z1 = arr[:, COL_Z] + 1.0
    a = 1.0 / z1
        # Background columns defined in the first code cell
    phi_smg   = arr[:, COL_PHI]      # phi_smg in unit of M_pl
    H0        = arr[-1, COL_H]
    phi_prime = arr[:, COL_PHI_PRIME]  # stored directly as dphi/deta
    # w_smg     = arr[:, 21] / arr[:, 20]  # w = p_smg/rho_smg
    phi_prime_prime = arr[:, COL_PHI_PP]  # stored directly as d^2phi/deta^2

    label = rf'$(\sigma={float(sigma_l):g},\,\alpha={float(alpha_l):g})$'

    # Left panel: scalar amplitude phi/M_P
    ax_phiH0.semilogx(z1, (phi_smg), '-', c=colors[num], lw=lw_f, label=label)

    # Middle panel: |phi'(z)| with sign encoded by linestyle
    phi_p = phi_prime
    pos = phi_p > 0
    neg = phi_p < 0

    ax_phiPrimeAbs.loglog(z1,  phi_p/H0,  '-',  c=colors[num], lw=lw_f)
    ax_phiPrimeAbs.loglog(z1, -phi_p/H0, '--', c=colors[num], lw=lw_f)

    # Right panel: w_phi(z)    
    # w_plot = w_smg.copy()
    # mask = (w_plot <= -1.25) | (w_plot >= 0.5)
    # w_plot[mask] = np.nan

    pos_p = phi_prime_prime > 0
    neg_p = phi_prime_prime < 0
    ax_phi_pp.semilogx(z1[pos_p],  phi_prime_prime[pos_p]/H0/H0, '-', c=colors[num], lw=lw_f)
    ax_phi_pp.semilogx(z1[neg_p],  -phi_prime_prime[neg_p]/H0/H0, '--', c=colors[num], lw=lw_f)
    # ax_phi_pp.axhline(-1.0, color='k', linestyle='-.', lw=1.0, alpha=0.6)
    # ax_phi_pp.axhline(0.0,  color='k', linestyle='-.', lw=1.0, alpha=0.3)
    # ax_phi_pp.axhline(1.0/3.0, color='k', linestyle='-.', lw=1.0, alpha=0.3)

# ---- Axes formatting ----
for ax in axes:
    ax.set_xlim(1, 1.e7)
    ax.grid(True, axis='both')

for ax in axes:
    ax.tick_params(which='both', direction='in', top=True, right=True)
    ax.grid(True, which='major', alpha=0.35)
    ax.grid(True, which='minor', alpha=0.15)

ax_phiH0.set_xlabel(r'$1+z$')
ax_phiH0.set_yscale('log')

ax_phiPrimeAbs.set_xlabel(r'$1+z$')
ax_phiPrimeAbs.set_yscale('log')

ax_phi_pp.set_xlabel(r'$1+z$')
ax_phi_pp.set_ylim(1.e-6, 1.e5)

ax_phiPrimeAbs.set_ylim(8.e-11, 3)
ax_phiH0.set_ylim(5.e-7, 5e5)

ax_phiH0.set_ylabel(r'$\varphi(z)/M_{\rm P}$')
ax_phiPrimeAbs.set_ylabel(r'$|\varphi^{\prime}(z)|/(H_0 M_{\rm P})$')
ax_phi_pp.set_ylabel(r'$|\varphi^{\prime \prime}(z)|/(H_0^2 M_{\rm P})$')
ax_phi_pp.set_yscale('log')

# Legend only on the left axis (model legend)
ax_phiH0.legend(loc='upper right', bbox_to_anchor=(1.0, 1.0),
                frameon=True, fontsize=22.5, borderaxespad=0.1)

# Sign legend for phi'
sign_legend = [
    Line2D([0], [0], color='k', lw=lw_f, linestyle='-',  label=r'$\varphi^\prime>0$'),
    Line2D([0], [0], color='k', lw=lw_f, linestyle='--', label=r'$\varphi^\prime<0$'),
]
ax_phiPrimeAbs.legend(handles=sign_legend, loc='upper right', frameon=True, fontsize=27)

sign_legend2 = [
    Line2D([0], [0], color='k', lw=lw_f, linestyle='-',  label=r'$\varphi^{\prime \prime}>0$'),
    Line2D([0], [0], color='k', lw=lw_f, linestyle='--', label=r'$\varphi^{\prime \prime}<0$'),
]
ax_phi_pp.legend(handles=sign_legend2, loc='upper left', frameon=True, fontsize=27)
# Save / show
plt.savefig('./Figs/phi_phi_prime_phi_pp.pdf', format='pdf', dpi=300,
            bbox_inches='tight', pad_inches=0.1)
plt.show()
